# F10: Month-End Close Simulation (Fabric Scenario)

## What You'll Learn

Month-end close is the heartbeat of financial reporting. Every month, new
transactions accumulate, adjusting entries are posted, and the books are
closed. Testing the pipelines and reports that support this process requires
data that **evolves over time** — not a static snapshot.

In this notebook you will:

1. **Understand** the month-end close scenario and its data requirements
2. **Generate** an initial financial dataset
3. **Create** 12 monthly snapshots using the time-travel engine
4. **Analyze** how data accumulates month over month
5. **Add** end-of-month adjustment entries using the continue engine
6. **Export** snapshots as partitioned Parquet for lakehouse loading

## The Month-End Close Pattern

```
Jan: Initial load       -> 10,000 transactions
Feb: Growth + adjust    -> 10,300 + 50 adjustments
Mar: Q1 close (spike)   -> 15,450 + 150 adjustments
...
Dec: Year-end close     -> 20,000 + 500 adjustments
```

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with T14 (incremental/time-travel patterns)

## Time Estimate

**~15 minutes**

In [1]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Step 1 — Generate the Initial Financial Dataset

**What we're about to do:** Generate a baseline financial dataset at small
scale. The financial domain includes accounts, journal entries, transactions,
cost centers, and GL balances.

**Why this matters:** This is your January 1st starting point — the opening
balances and initial transaction set that the time-travel engine will evolve
over 12 months. Think of it as the initial full load into your financial
data warehouse.

**What to expect:** A financial dataset with realistic account structures,
journal entries, and transaction records.

In [2]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.financial import FinancialDomain

domain = FinancialDomain()
baseline = Spindle().generate(domain=domain, scale="small", seed=42)

print("Financial Domain — Baseline (January)")
print("=" * 50)
for table_name, df in baseline.tables.items():
    print(f"  {table_name:<25} {len(df):>8,} rows")

total = sum(len(df) for df in baseline.tables.values())
print(f"\nTotal rows: {total:,}")
print(f"\nThis baseline will be evolved over 12 months.")

Financial Domain — Baseline (January)
  branch                         200 rows
  customer                     1,000 rows
  account                      2,200 rows
  card                         1,760 rows
  loan                           400 rows
  loan_payment                 4,800 rows
  statement                   13,200 rows
  transaction_category            40 rows
  transaction                 10,000 rows
  fraud_flag                     200 rows

Total rows: 33,800

This baseline will be evolved over 12 months.


## Step 2 — Configure Time-Travel for 12 Months

**What we're about to do:** Set up the `TimeTravelConfig` with 12 months of
evolution, including quarter-end spikes that simulate the increased activity
during financial close periods (March, June, September, December).

**Why this matters:** Financial data doesn't grow uniformly. Quarter-end and
year-end periods see dramatically higher transaction volumes due to accruals,
adjustments, reclassifications, and audit entries. The `seasonality` parameter
captures this pattern:
- **Month 3 (March):** 1.5x normal — Q1 close
- **Month 6 (June):** 1.3x normal — Q2 close
- **Month 9 (September):** 1.3x normal — Q3 close
- **Month 12 (December):** 2.0x normal — Year-end close

**What to expect:** A time-travel result containing 12 monthly snapshots with
realistic growth patterns and seasonal spikes.

In [3]:
from sqllocks_spindle.incremental.time_travel import TimeTravelEngine, TimeTravelConfig

tt_engine = TimeTravelEngine()
config = TimeTravelConfig(
    months=12,
    start_date="2024-01-01",
    growth_rate=0.03,          # 3% month-over-month base growth
    churn_rate=0.005,          # 0.5% monthly churn (reversals, voids)
    seasonality={
        3: 1.5,   # Q1 close — 50% spike
        6: 1.3,   # Q2 close — 30% spike
        9: 1.3,   # Q3 close — 30% spike
        12: 2.0,  # Year-end close — 100% spike
    },
    seed=42,
)

print("Time-Travel Configuration")
print("=" * 45)
print(f"  Months:       {config.months}")
print(f"  Start date:   {config.start_date}")
print(f"  Growth rate:  {config.growth_rate:.1%} per month")
print(f"  Churn rate:   {config.churn_rate:.1%} per month")
print(f"  Seasonality:")
for month, multiplier in config.seasonality.items():
    label = {3: 'Q1 close', 6: 'Q2 close', 9: 'Q3 close', 12: 'Year-end'}[month]
    print(f"    Month {month:>2} ({label}):  {multiplier:.1f}x")

Time-Travel Configuration
  Months:       12
  Start date:   2024-01-01
  Growth rate:  3.0% per month
  Churn rate:   0.5% per month
  Seasonality:
    Month  3 (Q1 close):  1.5x
    Month  6 (Q2 close):  1.3x
    Month  9 (Q3 close):  1.3x
    Month 12 (Year-end):  2.0x


## Step 3 — Generate 12 Monthly Snapshots

**What we're about to do:** Run the time-travel engine to produce 12 monthly
snapshots of the financial dataset. Each snapshot represents the state of the
data at month-end — as if you queried the warehouse on the last day of each month.

**Why this matters:** This is the data your month-end close reports and dashboards
will consume. Each snapshot includes all prior months' data plus the current
month's new transactions, giving you a complete cumulative history.

**What to expect:** A time-travel result with summary statistics showing the
growth trajectory across 12 months.

In [4]:
result = tt_engine.generate(domain=FinancialDomain(), config=config, scale="small")

print(result.summary())

Time-Travel Result
Domain: financial
Snapshots: 13

  Month           Date          Tables       Rows
  ----------------------------------------------
  Month 0        2024-01-01        10     33,800
  Month 1        2024-02-01        10     34,645
  Month 2        2024-03-01        10     36,029
  Month 3        2024-04-01        10     36,928
  Month 4        2024-05-01        10     37,849
  Month 5        2024-06-01        10     39,135
  Month 6        2024-07-01        10     40,112
  Month 7        2024-08-01        10     41,115
  Month 8        2024-09-01        10     42,511
  Month 9        2024-10-01        10     43,574
  Month 10       2024-11-01        10     44,662
  Month 11       2024-12-01        10     47,118
  Month 12       2025-01-01        10     48,296


## Step 4 — Month-Over-Month Accumulation

**What we're about to do:** Walk through each monthly snapshot and show how
the data accumulates — total rows, growth from prior month, and the effect
of quarter-end seasonality spikes.

**Why this matters:** Seeing the accumulation curve validates that the time-
travel engine is producing realistic financial data growth. The quarter-end
spikes should be clearly visible in the month-over-month deltas. This is
exactly the pattern you'd see in a production financial warehouse.

**What to expect:** Increasing row counts with visible spikes in March,
June, September, and December.

In [5]:
month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

if hasattr(result, 'snapshots'):
    print("=== Month-Over-Month Accumulation ===")
    print(f"{'Month':<8} {'Tables':>7} {'Total Rows':>12} {'Delta':>10} {'Growth':>8} {'Note':>12}")
    print("-" * 62)

    prev_rows = 0
    for i, snapshot in enumerate(result.snapshots):
        total_rows = sum(len(df) for df in snapshot.tables.values())
        delta = total_rows - prev_rows
        growth = f"{delta/prev_rows*100:.1f}%" if prev_rows > 0 else "-"
        month_num = i + 1
        note = ""
        if month_num in config.seasonality:
            note = f"{config.seasonality[month_num]:.1f}x spike"

        label = month_names[i] if i < len(month_names) else f"M{i+1}"
        print(f"  {label:<6} {len(snapshot.tables):>7} {total_rows:>12,} {delta:>+10,} {growth:>8} {note:>12}")
        prev_rows = total_rows

    first_rows = sum(len(df) for df in result.snapshots[0].tables.values())
    print(f"\nTotal growth: {first_rows:,} -> {prev_rows:,} ({(prev_rows/first_rows - 1)*100:.1f}% over 12 months)")
else:
    print("Time-travel result:")
    print(f"  Tables: {len(result.tables)}")
    print(f"  Total rows: {sum(len(df) for df in result.tables.values()):,}")

=== Month-Over-Month Accumulation ===
Month     Tables   Total Rows      Delta   Growth         Note
--------------------------------------------------------------
  Jan         10       33,800    +33,800        -             
  Feb         10       34,645       +845     2.5%             
  Mar         10       36,029     +1,384     4.0%   1.5x spike
  Apr         10       36,928       +899     2.5%             
  May         10       37,849       +921     2.5%             
  Jun         10       39,135     +1,286     3.4%   1.3x spike
  Jul         10       40,112       +977     2.5%             
  Aug         10       41,115     +1,003     2.5%             
  Sep         10       42,511     +1,396     3.4%   1.3x spike
  Oct         10       43,574     +1,063     2.5%             
  Nov         10       44,662     +1,088     2.5%             
  Dec         10       47,118     +2,456     5.5%   2.0x spike
  M13         10       48,296     +1,178     2.5%             

Total growth: 33

## Step 5 — End-of-Month Adjustment Entries

**What we're about to do:** Use the `ContinueEngine` to add adjustment
entries to a specific month's snapshot — simulating the accruals,
reclassifications, and corrections that happen during month-end close.

**Why this matters:** Month-end close isn't just about accumulating
transactions. Controllers post adjusting journal entries to correct
accruals, reclassify expenses, and reconcile accounts. These adjustments
are a critical part of the close process and your pipeline needs to
handle them correctly.

**What to expect:** A delta containing adjustment entries tagged with
`_delta_type` metadata.

In [6]:
from sqllocks_spindle.incremental import ContinueEngine, ContinueConfig

# Simulate adjustments for a quarter-end close (e.g., March)
adjust_engine = ContinueEngine()
adjust_config = ContinueConfig(
    insert_count=25,            # 25 adjusting journal entries
    update_fraction=0.05,       # 5% of existing entries get corrections
    delete_fraction=0.01,       # 1% reversals
    seed=42,
)

# Apply adjustments to the baseline
adjustments = adjust_engine.continue_from(baseline, config=adjust_config)

print("=== Month-End Adjustment Entries ===")
print(adjustments.summary())

# Show delta breakdown using adjustments.combined (not .tables)
print("\n=== Adjustment Breakdown ===")
for table_name, df in adjustments.combined.items():
    if "_delta_type" in df.columns and len(df) > 0:
        counts = df["_delta_type"].value_counts()
        parts = [f"{dtype}: {count}" for dtype, count in counts.items()]
        print(f"  {table_name:<25} {', '.join(parts)}")

=== Month-End Adjustment Entries ===
Incremental Generation Result
  branch: +25 inserts, ~10 updates, -2 deletes
  customer: +25 inserts, ~50 updates, -10 deletes
  account: +25 inserts, ~110 updates, -22 deletes
  card: +25 inserts, ~88 updates, -17 deletes
  loan: +25 inserts, ~20 updates, -4 deletes
  loan_payment: +25 inserts, ~240 updates, -48 deletes
  statement: +25 inserts, ~660 updates, -132 deletes
  transaction_category: +25 inserts, ~2 updates, -1 deletes
  transaction: +25 inserts, ~500 updates, -100 deletes
  fraud_flag: +25 inserts, ~10 updates, -2 deletes

=== Adjustment Breakdown ===
  branch                    INSERT: 25, UPDATE: 10, DELETE: 2
  customer                  UPDATE: 50, INSERT: 25, DELETE: 10
  account                   UPDATE: 110, INSERT: 25, DELETE: 22
  card                      UPDATE: 88, INSERT: 25, DELETE: 17
  loan                      INSERT: 25, UPDATE: 20, DELETE: 4
  loan_payment              UPDATE: 240, DELETE: 48, INSERT: 25
  statement  

C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\sqllocks_spindle\incremental\continue_engine.py:464: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(subset)
C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\sqllocks_spindle\incremental\continue_engine.py:464: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(subset)
C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\sqllocks_spindle\incremental\continue_engine.py:464: UserWarning: you are shuffling a 'ArrowStringArray' objec

## Step 6 — Export Snapshots as Partitioned Parquet

**What we're about to do:** Write each monthly snapshot to a date-partitioned
Parquet folder structure — the format that a Fabric Lakehouse or Delta Lake
expects for time-partitioned financial data.

**Why this matters:** Financial reporting requires point-in-time queries:
"What did the balance sheet look like at the end of March?" Partitioning
snapshots by month enables efficient time-travel queries and supports
SCD Type 2 patterns in your lakehouse.

**What to expect:** A folder per month, each containing Parquet files for
all financial tables.

In [7]:
import os

output_base = "./f10_month_end"

total_files = 0
total_size = 0

if hasattr(result, 'snapshots'):
    for i, snapshot in enumerate(result.snapshots):
        month_label = f"2024-{i+1:02d}"
        month_dir = os.path.join(output_base, month_label)
        os.makedirs(month_dir, exist_ok=True)

        for table_name, df in snapshot.tables.items():
            path = os.path.join(month_dir, f"{table_name}.parquet")
            df.to_parquet(path, index=False)
            total_files += 1
            total_size += os.path.getsize(path)

    print(f"=== Parquet Export Summary ===")
    print(f"  Output directory: {output_base}")
    print(f"  Monthly folders:  12")
    print(f"  Total files:      {total_files}")
    print(f"  Total size:       {total_size:,} bytes ({total_size/1024:.1f} KB)")

    # Show directory structure
    print(f"\n=== Directory Structure ===")
    for root, dirs, files in os.walk(output_base):
        level = root.replace(output_base, "").count(os.sep)
        indent = "  " * (level + 1)
        folder = os.path.basename(root)
        file_count = len(files)
        folder_size = sum(os.path.getsize(os.path.join(root, f)) for f in files)
        if level == 0:
            print(f"{indent}{folder}/")
        else:
            print(f"{indent}{folder}/  ({file_count} files, {folder_size:,} bytes)")
else:
    # Fallback: export the single result
    os.makedirs(output_base, exist_ok=True)
    for table_name, df in result.tables.items():
        path = os.path.join(output_base, f"{table_name}.parquet")
        df.to_parquet(path, index=False)
        total_files += 1
        total_size += os.path.getsize(path)
    print(f"Exported {total_files} files ({total_size:,} bytes)")

print(f"\nThese partitioned snapshots can be loaded into a Fabric Lakehouse")
print(f"for point-in-time financial reporting and month-end close dashboards.")

=== Parquet Export Summary ===
  Output directory: ./f10_month_end
  Monthly folders:  12
  Total files:      130
  Total size:       22,254,241 bytes (21732.7 KB)

=== Directory Structure ===
  f10_month_end/
    2024-01/  (10 files, 1,614,023 bytes)
    2024-02/  (10 files, 1,622,319 bytes)
    2024-03/  (10 files, 1,640,166 bytes)
    2024-04/  (10 files, 1,652,363 bytes)
    2024-05/  (10 files, 1,665,239 bytes)
    2024-06/  (10 files, 1,684,498 bytes)
    2024-07/  (10 files, 1,701,157 bytes)
    2024-08/  (10 files, 1,717,672 bytes)
    2024-09/  (10 files, 1,742,645 bytes)
    2024-10/  (10 files, 1,761,815 bytes)
    2024-11/  (10 files, 1,781,844 bytes)
    2024-12/  (10 files, 1,823,709 bytes)
    2024-13/  (10 files, 1,846,791 bytes)

These partitioned snapshots can be loaded into a Fabric Lakehouse
for point-in-time financial reporting and month-end close dashboards.


## What's Next?

You've simulated a full fiscal year of financial data — from January's opening
balances through December's year-end close, complete with quarter-end spikes
and adjusting entries.

| Notebook | What You'll Learn |
|----------|-------------------|
| **T14: Day 2 — Incremental** | Deep dive into CDC patterns and state transitions |
| **F09: Cross-Domain Enterprise** | Combine financial data with retail and HR for a full enterprise lakehouse |
| **F01: Medallion Architecture** | Build bronze/silver/gold layers from monthly snapshots |
| **F02: Warehouse Dimensional** | Create a dimensional model optimized for financial reporting |
| **[F04 — Real-Time Streaming](./F04_realtime_streaming.ipynb)** | Stream financial events in real time with burst windows and anomaly injection |
| **[F08 — File-Drop Ingestion](./F08_filedrop_ingestion.ipynb)** | Test file-drop pipelines with daily drops, late arrivals, and chaos injection |

**Key takeaways:**
- `TimeTravelEngine` produces 12 months of cumulative financial snapshots in seconds
- `seasonality` parameter captures quarter-end and year-end volume spikes
- `ContinueEngine` adds realistic adjusting entries for month-end close simulation
- Date-partitioned Parquet export maps directly to Fabric Lakehouse structure
- Deterministic seeds ensure reproducible financial data for regression testing